# Rescore the Final Tuning Eval Run on the Corrected Benchmark

Owner benchmark review, build 1 (2026-11-26). Eleven benchmark questions were corrected after tuning converged (wave 1-13); the benchmark stayed frozen during tuning. This notebook re-grades the final tuning eval run offline for the questions whose expectation changed, using Genie's chat-mode rules, and reports the frozen and corrected scores per split. Questions whose wording changed need a new evaluation and are not rescored. Results: [benchmark_review.md](../benchmark_review.md).

In [0]:
import os
import subprocess
import sys
import tempfile
import urllib.request

SDK = "/Workspace/Users/matthew.giglia@databricks.com/mbp/genie-ontology-modeled-knowledge-sdk"
REPO = os.path.dirname(SDK)
sys.path.insert(0, SDK + "/src")
from genie_ontology_modeled_knowledge_sdk import benchmark as bm

BENCH_REL = "genie-ontology-modeled-knowledge-sdk/examples/tpch/docs/tuning/benchmark"
FROZEN_COMMIT = "1185bfe58e11087adfa4b2e0ef9394a29edc66f7"  # last commit before the corrections (benchmark frozen since the baseline)
RAW = "https://raw.githubusercontent.com/mkgs-databricks-demos/mbp"
CATALOG, SCHEMA = "hls_fde_dev", "dev_matthew_giglia_ontology"

def resolve(sql: str) -> str:
    return sql.replace("${target_catalog}", CATALOG).replace("${target_schema}", SCHEMA)

tmp = tempfile.mkdtemp()
for f in sorted(os.listdir(os.path.join(REPO, BENCH_REL))):
    if f.endswith(".yml"):
        # git on the workspace file system can fail on some compute (SIGBUS); fall back to the public repo at the same commit.
        res = subprocess.run(["git", "-C", REPO, "show", f"{FROZEN_COMMIT}:{BENCH_REL}/{f}"], capture_output=True, text=True)
        txt = res.stdout if res.returncode == 0 else urllib.request.urlopen(f"{RAW}/{FROZEN_COMMIT}/{BENCH_REL}/{f}").read().decode()
        open(os.path.join(tmp, f), "w").write(txt)
old = {q.id: q for q in bm.load_question_set(tmp)}
new = {q.id: q for q in bm.load_question_set(os.path.join(REPO, BENCH_REL))}
splits = bm.assign_splits(list(new.values()))
assert splits == bm.assign_splits(list(old.values())), "split assignment changed"

changed = sorted(i for i in new if (old[i].question, (old[i].expected_sql or "").strip()) != (new[i].question, (new[i].expected_sql or "").strip()))
reworded = {i for i in changed if old[i].question != new[i].question}
print(len(new), "questions; changed:", changed, "| reworded (need a new eval):", sorted(reworded))

In [0]:
import json
from databricks.sdk import WorkspaceClient

ws = WorkspaceClient()
SPACE_ID = "01f1e8a428241105b01bbc2f4e355e30"  # [dev matthew_giglia] TPC-H Supply Chain Agent
history = json.load(open(SDK + "/examples/tpch/docs/tuning/accuracy_history.json"))
final = history[-1]
EVAL_RUN = final["eval_run_id"]
print("final tuning iteration:", final["iteration"], final["label"], "eval run", EVAL_RUN)

base = f"/api/2.0/genie/spaces/{SPACE_ID}/eval-runs/{EVAL_RUN}/results"
by_bid = {bm.benchmark_id(q.id): q for q in new.values()}
items, token = [], None
while True:
    page = bm._api(ws, "GET", base, query={"page_size": 100, **({"page_token": token} if token else {})})
    items += page.get("eval_results") or []
    token = page.get("next_page_token")
    if not token:
        break

frozen = {}  # question id -> (Genie assessment, generated SQL)
for it in items:
    q = by_bid.get(str(it.get("benchmark_question_id")))
    if q is None:
        continue
    d = bm._api(ws, "GET", f"{base}/{it['result_id']}")
    sql = next((r.get("response") for r in d.get("actual_response") or [] if r.get("response_type") == "SQL"), None)
    frozen[q.id] = (d.get("assessment"), sql)

def split_score(verdicts):
    out = {}
    for split in ("tuning", "holdout"):
        ids = [i for i in verdicts if splits.get(i) == split]
        out[split] = (sum(1 for i in ids if verdicts[i] == "GOOD"), len(ids))
    return out

frozen_verdicts = {i: a for i, (a, _) in frozen.items()}
print("mapped:", len(frozen), "| frozen score:", split_score(frozen_verdicts),
      "| accuracy_history:", (final["tuning"]["correct"], final["tuning"]["n"]), (final["holdout"]["correct"], final["holdout"]["n"]))

In [0]:
import math
from collections import Counter
from decimal import Decimal

def _sig4(v):
    if v is None:
        return None
    if isinstance(v, (int, float, Decimal)) and not isinstance(v, bool):
        f = float(v)
        return 0.0 if f == 0 or math.isnan(f) else float(f"{f:.4g}")
    try:
        return float(f"{float(str(v)):.4g}")
    except ValueError:
        return str(v).strip()

def _rows(sql):
    df = spark.sql(sql)
    return df.columns, Counter(tuple(_sig4(x) for x in r) for r in df.collect())

def genie_match(generated_sql, expected_sql):
    """Strict reading of the chat-mode rules: same column count, same rows in any order, numbers to 4 significant digits."""
    if not generated_sql:
        return False, "no SQL generated"
    try:
        gcols, grows = _rows(generated_sql)
    except Exception as e:  # noqa: BLE001
        return False, f"generated SQL failed: {str(e)[:120]}"
    ecols, erows = _rows(resolve(expected_sql))
    if not grows:
        return False, "empty generated result"
    if len(gcols) != len(ecols):
        return False, f"{len(gcols)} columns vs {len(ecols)} expected ({', '.join(gcols)})"
    return (grows == erows), ("rows match" if grows == erows else "rows differ")

# The offline grader must not be more lenient than Genie: check it on every unchanged question.
agree, disagree = 0, []
for qid, (assessment, sql) in sorted(frozen.items()):
    if qid in changed or assessment not in ("GOOD", "BAD"):
        continue
    ok, why = genie_match(sql, new[qid].expected_sql)
    if ok == (assessment == "GOOD"):
        agree += 1
    else:
        disagree.append((qid, assessment, why))
print(f"grader vs Genie on unchanged questions: agree {agree}, disagree {len(disagree)}; disagreements by Genie verdict: {Counter(a for _, a, _ in disagree)}")

In [0]:
rescored, rows = dict(frozen_verdicts), []
for qid in changed:
    before = frozen_verdicts.get(qid)
    if qid in reworded:
        rows.append((qid, splits.get(qid), before, "not rescored", "question reworded: needs a new evaluation"))
        continue
    ok, why = genie_match(frozen[qid][1], new[qid].expected_sql)
    rescored[qid] = "GOOD" if ok else "BAD"
    rows.append((qid, splits.get(qid), before, rescored[qid], why))

display(spark.createDataFrame(rows, "id string, split string, frozen string, corrected string, reason string"))
print("frozen benchmark   :", split_score(frozen_verdicts))
print("corrected benchmark:", split_score(rescored))
print("corrected, reworded questions excluded:", split_score({k: v for k, v in rescored.items() if k not in reworded}))